In [1]:
# %%
# LightGCN for Recommendation — Sparse / Memory-Efficient Version
# Clean corrected version

import math
import os
import random
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

try:
    import psutil
    HAVE_PSUTIL = True
except ImportError:
    HAVE_PSUTIL = False
    print("psutil not found — install with: pip install psutil")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# %%
config_dict = {
    "num_samples_per_user": 10, # 1 to 50 (increased by 5)
    "num_users": None,

    "epochs": 50, # 50 for MovieLens, 10 for Amazon Beauty
    "batch_size": 4096, # 1024 for MovieLens, 4096 for Amazon Beauty
    "eval_batch_size": 1024,   # 512 for MovieLens, 1024 for Amazon Beauty
    "lr": 0.01,
    "weight_decay": 0.1,

    "embedding_size": 64,
    "num_layers": 1, # 1-3 layers for MovieLens, 1 layer for Amazon Beauty
    "K": 10,

    "minibatch_per_print": 100,
    "epochs_per_print": 5,

    "val_frac": 0.1,
    "test_frac": 0.2,
    "seed": 42,

    # "movielens100k" | "movielens1m" | "amazonBeauty2m"
    "dataset_name": "amazonBeauty2m",
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(config_dict["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

base_path = {
    "movielens100k":  "log/ml_100k/",
    "movielens1m":    "log/ml_1m/",
    "amazonBeauty2m": "log/beauty2m/",
}

output_file_name = (
    base_path[config_dict["dataset_name"]]
    + f"result_{config_dict['dataset_name']}_"
    + f"{config_dict['num_layers']}layers_"
    + f"{config_dict['num_samples_per_user']}sample_"
    + f"{config_dict['num_users'] if config_dict['num_users'] is not None else 'all'}users.pkl"
)
print("Output file:", output_file_name)

# %%
# ---------------------------------------------------------------------------
# Memory utilities
# ---------------------------------------------------------------------------

def get_ram_usage_mb() -> float:
    if HAVE_PSUTIL:
        return psutil.Process(os.getpid()).memory_info().rss / 1024 ** 2
    return 0.0

def get_gpu_mb():
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024 ** 2
        reserved = torch.cuda.memory_reserved() / 1024 ** 2
        peak = torch.cuda.max_memory_allocated() / 1024 ** 2
        return alloc, reserved, peak
    return 0.0, 0.0, 0.0

def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def sparse_csr_mem_mb(csr) -> float:
    return (csr.data.nbytes + csr.indices.nbytes + csr.indptr.nbytes) / 1024 ** 2

def dict_of_arrays_mem_mb(d: dict) -> float:
    return sum(v.nbytes for v in d.values()) / 1024 ** 2

def torch_sparse_mem_mb(t: torch.Tensor) -> float:
    return (
        t._indices().element_size() * t._indices().nelement()
        + t._values().element_size() * t._values().nelement()
    ) / 1024 ** 2

def torch_param_mem_mb(model: torch.nn.Module) -> float:
    return sum(p.element_size() * p.nelement() for p in model.parameters()) / 1024 ** 2

def theoretical_mem_mb(U, I, R_nnz, d, L) -> dict:
    f32, i64 = 4, 8

    emb_bytes = (U + I) * d * f32
    adj_val_bytes = 2 * R_nnz * f32
    adj_idx_bytes = 2 * 2 * R_nnz * i64
    prop_bytes = (L + 1) * (U + I) * d * f32
    adam_bytes = 2 * emb_bytes
    grad_bytes = emb_bytes

    total = emb_bytes + adj_val_bytes + adj_idx_bytes + prop_bytes + adam_bytes + grad_bytes
    MB = 1024 ** 2

    return {
        "embeddings_MB": emb_bytes / MB,
        "adj_values_MB": adj_val_bytes / MB,
        "adj_indices_MB": adj_idx_bytes / MB,
        "propagation_MB": prop_bytes / MB,
        "adam_moments_MB": adam_bytes / MB,
        "gradients_MB": grad_bytes / MB,
        "total_MB": total / MB,
    }

class MemSnapshot:
    def __init__(self, tag: str):
        self.tag = tag
        self.ram_mb = get_ram_usage_mb()
        self.gpu_alloc_mb, self.gpu_reserved_mb, self.gpu_peak_mb = get_gpu_mb()

    def __repr__(self):
        return (
            f"[MEM | {self.tag:40s}] "
            f"RAM={self.ram_mb:8.1f} MB | "
            f"GPU alloc={self.gpu_alloc_mb:8.1f} MB | "
            f"GPU peak={self.gpu_peak_mb:8.1f} MB"
        )

def snap(tag: str) -> MemSnapshot:
    s = MemSnapshot(tag)
    print(s)
    return s

mem_log = []

# %%
# ---------------------------------------------------------------------------
# Dataset loaders
# ---------------------------------------------------------------------------

class MovieLens100K:
    def __init__(self):
        path = "datasets/ml_100k/u.data"
        df = pd.read_csv(path, sep="\t", names=["uid", "iid", "rating", "ts"])
        self._build(df, "uid", "iid")

    def _build(self, df, ucol, icol):
        self.user_ids = np.sort(df[ucol].unique())
        self.item_ids = np.sort(df[icol].unique())
        u2i = {u: i for i, u in enumerate(self.user_ids)}
        i2i = {m: i for i, m in enumerate(self.item_ids)}
        rows = df[ucol].map(u2i).to_numpy()
        cols = df[icol].map(i2i).to_numpy()
        vals = np.ones(len(df), dtype=np.float32)
        nu, ni = len(self.user_ids), len(self.item_ids)
        print(f"n_users:{nu:,}  n_items:{ni:,}  interactions:{len(df):,}")
        self.R = sp.csr_matrix((vals, (rows, cols)), shape=(nu, ni))

    def get_interaction_matrix(self):
        return self.R

class MovieLens1M:
    def __init__(self):
        path = "datasets/ml_1m/ratings.dat"
        df = pd.read_csv(
            path,
            sep="::",
            names=["uid", "iid", "rating", "ts"],
            engine="python",
            encoding="latin-1"
        )
        self._build(df, "uid", "iid")

    def _build(self, df, ucol, icol):
        self.user_ids = np.sort(df[ucol].unique())
        self.item_ids = np.sort(df[icol].unique())
        u2i = {u: i for i, u in enumerate(self.user_ids)}
        i2i = {m: i for i, m in enumerate(self.item_ids)}
        rows = df[ucol].map(u2i).to_numpy()
        cols = df[icol].map(i2i).to_numpy()
        vals = np.ones(len(df), dtype=np.float32)
        nu, ni = len(self.user_ids), len(self.item_ids)
        print(f"n_users:{nu:,}  n_items:{ni:,}  interactions:{len(df):,}")
        self.R = sp.csr_matrix((vals, (rows, cols)), shape=(nu, ni))

    def get_interaction_matrix(self):
        return self.R

class AmazonBeauty2M:
    def __init__(self):
        path = "datasets/amazon_beauty/ratings_Beauty.csv"
        df = pd.read_csv(path).rename(columns={"UserId": "uid", "ProductId": "iid"})
        self._build(df, "uid", "iid")

    def _build(self, df, ucol, icol):
        self.user_ids = np.sort(df[ucol].unique())
        self.item_ids = np.sort(df[icol].unique())
        u2i = {u: i for i, u in enumerate(self.user_ids)}
        i2i = {m: i for i, m in enumerate(self.item_ids)}
        rows = df[ucol].map(u2i).to_numpy()
        cols = df[icol].map(i2i).to_numpy()
        vals = np.ones(len(df), dtype=np.float32)
        nu, ni = len(self.user_ids), len(self.item_ids)
        print(f"n_users:{nu:,}  n_items:{ni:,}  interactions:{len(df):,}")
        self.R = sp.csr_matrix((vals, (rows, cols)), shape=(nu, ni))

    def get_interaction_matrix(self):
        return self.R

DATASETS = {
    "movielens100k": MovieLens100K,
    "movielens1m": MovieLens1M,
    "amazonBeauty2m": AmazonBeauty2M,
}

snap_start = snap("startup (before anything)")
dataset = DATASETS[config_dict["dataset_name"]]()
R_all = dataset.get_interaction_matrix().tocsr()

if config_dict["num_users"] is not None:
    R_all = R_all[:config_dict["num_users"], :]

n_users, n_items = R_all.shape
print(f"#Users: {n_users:,} | #Items: {n_items:,} | Interactions: {R_all.nnz:,}")

snap_after_load = snap("after dataset load")
r_all_mb = sparse_csr_mem_mb(R_all)
print(f"R_all CSR actual RAM   : {r_all_mb:.2f} MB")
print(f"Dense equivalent would : {n_users * n_items * 4 / 1024**3:.1f} GB")

# %%
# ---------------------------------------------------------------------------
# Sparse train / val / test split
# ---------------------------------------------------------------------------

def build_sparse_splits(R_csr, val_frac=0.1, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_pos, val_pos, test_pos = {}, {}, {}

    for u in range(R_csr.shape[0]):
        items = R_csr[u].indices.copy()
        n_pos = len(items)
        if n_pos == 0:
            continue

        rng.shuffle(items)
        n_val = int(n_pos * val_frac)
        n_test = int(n_pos * test_frac)

        if n_pos >= 3:
            n_val = max(n_val, 1) if val_frac > 0 else 0
            n_test = max(n_test, 1) if test_frac > 0 else 0

        if n_val + n_test >= n_pos:
            if n_pos >= 3:
                n_test = min(n_test, n_pos - 2)
                n_val = min(n_val, n_pos - n_test - 1)
            elif n_pos == 2:
                n_val, n_test = 1, 0
            else:
                n_val, n_test = 0, 0

        val_items = items[:n_val]
        test_items = items[n_val:n_val + n_test]
        train_items = items[n_val + n_test:]

        if len(train_items) == 0:
            train_items = items[-1:]
            if len(val_items) > 0:
                val_items = val_items[:-1]

        train_pos[u] = train_items
        if len(val_items) > 0:
            val_pos[u] = val_items
        if len(test_items) > 0:
            test_pos[u] = test_items

    return train_pos, val_pos, test_pos

train_pos, val_pos, test_pos = build_sparse_splits(
    R_all,
    val_frac=config_dict["val_frac"],
    test_frac=config_dict["test_frac"],
    seed=config_dict["seed"],
)

train_nnz = sum(len(v) for v in train_pos.values())
val_nnz = sum(len(v) for v in val_pos.values())
test_nnz = sum(len(v) for v in test_pos.values())

print(f"Train positives: {train_nnz:,}")
print(f"Val positives  : {val_nnz:,}")
print(f"Test positives : {test_nnz:,}")

train_mb = dict_of_arrays_mem_mb(train_pos)
val_mb = dict_of_arrays_mem_mb(val_pos)
test_mb = dict_of_arrays_mem_mb(test_pos)
total_split_mb = train_mb + val_mb + test_mb

print(f"Split dict RAM: train={train_mb:.2f} MB | val={val_mb:.2f} MB | test={test_mb:.2f} MB")
print(f"Total split RAM: {total_split_mb:.2f} MB")

snap_after_split = snap("after train/val/test split")

# -------------------------------------------------
# Dataset / training statistics (before training)
# -------------------------------------------------

num_users_total = n_users
num_items_total = n_items

# Users that actually have training interactions
num_train_users = len(train_pos)

# Total training edges
num_train_edges = sum(len(v) for v in train_pos.values())

# Total sampled triplets per epoch
k = config_dict["num_samples_per_user"]
num_triplets_per_epoch = num_train_users * k

# Number of batches
batch_size = config_dict["batch_size"]
num_batches_per_epoch = math.ceil(num_triplets_per_epoch / batch_size)

print("\n===== DATA / TRAINING STATS =====")
print(f"Total users                 : {num_users_total:,}")
print(f"Total items                 : {num_items_total:,}")
print(f"Train users (active)        : {num_train_users:,}")
print(f"Train edges (R_train.nnz)   : {num_train_edges:,}")
print(f"Samples per user (k)        : {k}")
print(f"Triplets per epoch          : {num_triplets_per_epoch:,}")
print(f"Batch size                  : {batch_size}")
print(f"Batches per epoch           : {num_batches_per_epoch}")
print("================================\n")

# %%
def build_train_csr(train_pos, n_users, n_items):
    n_total = sum(len(items) for items in train_pos.values())
    rows = np.empty(n_total, dtype=np.int32)
    cols = np.empty(n_total, dtype=np.int32)

    ptr = 0
    for u, items in train_pos.items():
        n = len(items)
        rows[ptr:ptr + n] = u
        cols[ptr:ptr + n] = items
        ptr += n

    vals = np.ones(n_total, dtype=np.float32)
    return sp.csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

R_train = build_train_csr(train_pos, n_users, n_items)
r_train_mb = sparse_csr_mem_mb(R_train)
print(f"Train graph nnz: {R_train.nnz:,} | CSR RAM: {r_train_mb:.2f} MB")

# %%
# ---------------------------------------------------------------------------
# LightGCN model
# ---------------------------------------------------------------------------

def normalize_sparse_matrix(R):
    U, I = R.shape
    N = U + I

    R_coo = R.tocoo()
    rows_R = R_coo.row.astype(np.int32)
    cols_R = R_coo.col.astype(np.int32)
    data_R = R_coo.data.astype(np.float32)

    all_row = np.concatenate([rows_R, cols_R + U])
    all_col = np.concatenate([cols_R + U, rows_R])
    all_data = np.concatenate([data_R, data_R])

    A = sp.csr_matrix((all_data, (all_row, all_col)), shape=(N, N), dtype=np.float32)
    deg = np.asarray(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(deg > 0, deg ** (-0.5), 0.0).astype(np.float32)
    D = sp.diags(d_inv_sqrt, format="csr")

    return D.dot(A).dot(D).tocsr()

def scipy_csr_to_torch_sparse(csr, device):
    coo = csr.tocoo()
    indices = torch.tensor(np.vstack([coo.row, coo.col]), dtype=torch.long, device=device)
    values = torch.tensor(coo.data, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(indices, values, size=coo.shape, device=device).coalesce()

class LightGCN(torch.nn.Module):
    def __init__(self, R_csr, embedding_dim=64, n_layers=3, seed=42, device="cpu"):
        super().__init__()
        torch.manual_seed(seed)
        np.random.seed(seed)

        self.R = R_csr.tocsr()
        self.num_users, self.num_items = self.R.shape
        self.emb_dim = embedding_dim
        self.n_layers = n_layers
        self.device = device

        self.all_embeddings = torch.nn.Embedding(self.num_users + self.num_items, self.emb_dim)
        torch.nn.init.normal_(self.all_embeddings.weight, mean=0.0, std=0.1)

        A_hat = normalize_sparse_matrix(self.R)
        self.norm_adj = scipy_csr_to_torch_sparse(A_hat, device=device)

        self.to(device)

    def propagate(self, return_time=False):
        t0 = time.perf_counter()
        X0 = self.all_embeddings.weight
        outs = [X0]
        X = X0

        for _ in range(self.n_layers):
            X = torch.sparse.mm(self.norm_adj, X)
            outs.append(X)

        X_final = torch.stack(outs, dim=0).mean(dim=0)
        user_emb = X_final[:self.num_users]
        item_emb = X_final[self.num_users:]

        if return_time:
            return user_emb, item_emb, time.perf_counter() - t0
        return user_emb, item_emb

# %%
# ---------------------------------------------------------------------------
# BPR loss
# ---------------------------------------------------------------------------

def get_embeddings(model, users, pos, neg, return_prop_time=False):
    if return_prop_time:
        user_final, item_final, prop_time = model.propagate(return_time=True)
    else:
        user_final, item_final = model.propagate()
        prop_time = 0.0

    n_user = model.num_users
    users_emb = user_final[users]
    pos_emb = item_final[pos]
    neg_emb = item_final[neg]

    users_emb_ego = model.all_embeddings(users)
    pos_emb_ego = model.all_embeddings(pos + n_user)
    neg_emb_ego = model.all_embeddings(neg + n_user)

    if return_prop_time:
        return users_emb, pos_emb, neg_emb, users_emb_ego, pos_emb_ego, neg_emb_ego, prop_time

    return users_emb, pos_emb, neg_emb, users_emb_ego, pos_emb_ego, neg_emb_ego

def bpr_loss(model, users, pos, neg, return_times=False):
    t0 = time.perf_counter()
    users, pos, neg = users.long(), pos.long(), neg.long()

    if return_times:
        u_e, p_e, n_e, u0, p0, n0, prop_time = get_embeddings(
            model, users, pos, neg, return_prop_time=True
        )
    else:
        u_e, p_e, n_e, u0, p0, n0 = get_embeddings(model, users, pos, neg)
        prop_time = 0.0

    score_time_start = time.perf_counter()

    reg_loss = 0.5 * (
        u0.norm(2).pow(2) + p0.norm(2).pow(2) + n0.norm(2).pow(2)
    ) / float(len(users))

    loss = torch.mean(
        torch.nn.functional.softplus(
            (u_e * n_e).sum(dim=1) - (u_e * p_e).sum(dim=1)
        )
    )

    scoring_time = time.perf_counter() - score_time_start
    total_forward_time = time.perf_counter() - t0

    if return_times:
        return loss, reg_loss, prop_time, scoring_time, total_forward_time

    return loss, reg_loss

# %%
# ---------------------------------------------------------------------------
# Negative sampling helpers
# ---------------------------------------------------------------------------

def build_exclusion_sets(*dicts):
    merged = {}
    all_users = set()
    for d in dicts:
        all_users |= set(d.keys())

    for u in all_users:
        parts = []
        for d in dicts:
            if u in d and len(d[u]) > 0:
                parts.append(d[u])
        if parts:
            merged[u] = np.unique(np.concatenate(parts))
    return merged

def sample_bpr_triplets_sparse(pos_dict, n_items, num_samples_per_user, seed=42, exclusion_dict=None):
    rng = np.random.default_rng(seed)
    samples = []

    for u, pos_items in pos_dict.items():
        if len(pos_items) == 0:
            continue

        excluded = exclusion_dict.get(u, pos_items) if exclusion_dict is not None else pos_items
        excluded_set = set(excluded.tolist())

        if len(excluded_set) >= n_items:
            continue

        for _ in range(num_samples_per_user):
            pos = int(rng.choice(pos_items))
            neg = int(rng.integers(0, n_items))
            while neg in excluded_set:
                neg = int(rng.integers(0, n_items))
            samples.append([u, pos, neg])

    if not samples:
        return torch.empty((0, 3), dtype=torch.long)

    return torch.tensor(samples, dtype=torch.long)

# %%
# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------

def compute_metrics_sparse(model, eval_pos, mask_pos, n_users, K=10, eval_batch_size=512, device="cpu"):
    model.eval()
    with torch.no_grad():
        user_final, item_final = model.propagate()

    precisions, recalls = [], []

    for start in range(0, n_users, eval_batch_size):
        end = min(start + eval_batch_size, n_users)
        batch_users = torch.arange(start, end, device=device)

        scores = user_final[batch_users] @ item_final.T

        for local_i, u in enumerate(range(start, end)):
            gt = eval_pos.get(u)
            if gt is None or len(gt) == 0:
                continue

            seen = mask_pos.get(u)
            if seen is not None and len(seen) > 0:
                scores[local_i, seen] = -1e9

            topk = torch.topk(scores[local_i], K).indices.cpu().numpy()
            hits = len(set(topk) & set(gt.tolist()))

            precisions.append(hits / K)
            recalls.append(hits / len(gt))

    p = float(np.mean(precisions)) if precisions else 0.0
    r = float(np.mean(recalls)) if recalls else 0.0
    return p, r

# %%
# ---------------------------------------------------------------------------
# Build model + optimizer
# ---------------------------------------------------------------------------

reset_gpu_peak()
snap_pre_model = snap("before model init")

model = LightGCN(
    R_train,
    embedding_dim=config_dict["embedding_size"],
    n_layers=config_dict["num_layers"],
    seed=config_dict["seed"],
    device=device,
)

snap_post_model = snap("after model init")

optimizer = torch.optim.Adam(model.parameters(), lr=config_dict["lr"])
snap_post_optim = snap("after optimizer init")

train_val_pos = build_exclusion_sets(train_pos, val_pos)
all_observed_pos = build_exclusion_sets(train_pos, val_pos, test_pos)

param_mb = torch_param_mem_mb(model)
adj_mb = torch_sparse_mem_mb(model.norm_adj)
theory = theoretical_mem_mb(
    n_users,
    n_items,
    R_train.nnz,
    config_dict["embedding_size"],
    config_dict["num_layers"],
)

print("\n" + "=" * 65)
print("STATIC MEMORY BREAKDOWN")
print("=" * 65)
print(f"Embedding table (params)        : {param_mb:8.2f} MB")
print(f"Norm adjacency (GPU sparse COO) : {adj_mb:8.2f} MB")
print(f"R_all CSR (CPU)                 : {r_all_mb:8.2f} MB")
print(f"R_train CSR (CPU)               : {r_train_mb:8.2f} MB")
print(f"Split dict total                : {total_split_mb:8.2f} MB")
print()
for k_name, v in theory.items():
    print(f"{k_name:36s}: {v:8.2f} MB")
print("=" * 65 + "\n")

# %%
# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

epochs_tracked = []
train_metrics = []
val_metrics = []

total_train_start = time.perf_counter()
total_eval_time = 0.0
total_prop_time = 0.0
total_scoring_time = 0.0
total_forward_time = 0.0
total_backward_step_time = 0.0
epoch_times = []

reset_gpu_peak()

for epoch in range(config_dict["epochs"]):
    print(f"Training on epoch {epoch}")

    epoch_start = time.perf_counter()
    model.train()
    loss_sum = 0.0

    samples_train = sample_bpr_triplets_sparse(
        pos_dict=train_pos,
        n_items=n_items,
        num_samples_per_user=config_dict["num_samples_per_user"],
        seed=config_dict["seed"] + epoch,
        exclusion_dict=train_pos,
    ).to(device)

    num_batches = math.ceil(len(samples_train) / config_dict["batch_size"]) if len(samples_train) > 0 else 0

    if len(samples_train) > 0:
        perm = torch.randperm(samples_train.size(0), device=device)
        samples_train = samples_train[perm]

    
    if epoch == 0:
        print(f"Actual sampled triplets this epoch: {len(samples_train):,}")

    for batch_idx in range(num_batches):
        optimizer.zero_grad()

        batch = samples_train[
            batch_idx * config_dict["batch_size"]:(batch_idx + 1) * config_dict["batch_size"]
        ]
        if len(batch) == 0:
            continue

        users_b, pos_b, neg_b = batch[:, 0], batch[:, 1], batch[:, 2]

        loss, reg_loss, prop_time, scoring_time, forward_time = bpr_loss(
            model, users_b, pos_b, neg_b, return_times=True
        )

        reg_loss_w = reg_loss * config_dict["weight_decay"]
        total_loss = loss + reg_loss_w

        total_prop_time += prop_time
        total_scoring_time += scoring_time
        total_forward_time += forward_time
        loss_sum += total_loss.detach().item()

        step_s = time.perf_counter()
        total_loss.backward()
        optimizer.step()
        total_backward_step_time += time.perf_counter() - step_s

        if batch_idx % config_dict["minibatch_per_print"] == 0:
            print(
                f"  Minibatch {batch_idx+1}/{num_batches} | "
                f"BPR={loss.item():.6f} | Reg={reg_loss_w.item():.6f}"
            )

    if epoch % config_dict["epochs_per_print"] == 0:
        eval_s = time.perf_counter()
        epochs_tracked.append(epoch)

        train_p, train_r = compute_metrics_sparse(
            model,
            eval_pos=train_pos,
            mask_pos={},  # keep as train-fit diagnostic
            n_users=n_users,
            K=config_dict["K"],
            eval_batch_size=config_dict["eval_batch_size"],
            device=device,
        )
        train_metrics.append((train_p, train_r))

        val_p, val_r = compute_metrics_sparse(
            model,
            eval_pos=val_pos,
            mask_pos=train_pos,
            n_users=n_users,
            K=config_dict["K"],
            eval_batch_size=config_dict["eval_batch_size"],
            device=device,
        )
        val_metrics.append((val_p, val_r))

        total_eval_time += time.perf_counter() - eval_s

        avg_loss = loss_sum / max(num_batches, 1)
        print(
            f"\nEpoch {epoch} | Loss={avg_loss:.6f} | "
            f"Train P@{config_dict['K']}={train_p:.4f} R@{config_dict['K']}={train_r:.4f} | "
            f"Val P@{config_dict['K']}={val_p:.4f} R@{config_dict['K']}={val_r:.4f}"
        )

    gpu_alloc, gpu_res, gpu_peak = get_gpu_mb()
    ram_now = get_ram_usage_mb()
    mem_log.append({
        "epoch": epoch,
        "ram_mb": ram_now,
        "gpu_alloc_mb": gpu_alloc,
        "gpu_reserved_mb": gpu_res,
        "gpu_peak_mb": gpu_peak,
    })

    epoch_time = time.perf_counter() - epoch_start
    epoch_times.append(epoch_time)
    print(
        f"Epoch {epoch} took {epoch_time:.4f}s | "
        f"GPU alloc={gpu_alloc:.1f} MB peak={gpu_peak:.1f} MB RAM={ram_now:.1f} MB\n"
    )

total_training_time = time.perf_counter() - total_train_start

print("\n===== Timing Summary =====")
print(f"Total training time    : {total_training_time:.4f}s")
print(f"Propagation time       : {total_prop_time:.4f}s")
print(f"Score/loss time        : {total_scoring_time:.4f}s")
print(f"Forward total time     : {total_forward_time:.4f}s")
print(f"Backward/step time     : {total_backward_step_time:.4f}s")
print(f"Evaluation time        : {total_eval_time:.4f}s")
print(f"Average epoch time     : {np.mean(epoch_times):.4f}s")

# %%
# ---------------------------------------------------------------------------
# Complexity summary
# ---------------------------------------------------------------------------

R_train_edges = R_train.nnz
k_cfg = config_dict["num_samples_per_user"]
L = config_dict["num_layers"]
d = config_dict["embedding_size"]
U = n_users
I = n_items

print("\n===== Complexity Parameters =====")
print(f"|U| = {U:,}")
print(f"|I| = {I:,}")
print(f"|R_train| = {R_train_edges:,}")
print(f"k = {k_cfg} | L = {L} | d = {d}")

print("\n===== Theoretical Complexity =====")
print("Propagation           ~ O(L |R| d)")
print("BPR objective         ~ O(k |U| d)")
print("Total training        ~ O(L |R| d + k |U| d)")
print("Space                 ~ O((|U| + |I|) d + |R|)")

# %%
# ---------------------------------------------------------------------------
# Training + memory plots
# ---------------------------------------------------------------------------

log_epochs = [e["epoch"] for e in mem_log]
log_ram = [e["ram_mb"] for e in mem_log]
log_gpu = [e["gpu_alloc_mb"] for e in mem_log]
log_peak = [e["gpu_peak_mb"] for e in mem_log]
epochs_arr = np.array(epochs_tracked)

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle(
    f"LightGCN — {config_dict['dataset_name']} | "
    f"L={L} | d={d} | k={k_cfg} | users={n_users:,}",
    fontsize=13,
)

ax = axes[0, 0]
ax.plot(epochs_arr, [p for p, _ in train_metrics], label="Train")
ax.plot(epochs_arr, [p for p, _ in val_metrics], label="Val")
ax.set_title(f"Precision@{config_dict['K']}")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(epochs_arr, [r for _, r in train_metrics], label="Train")
ax.plot(epochs_arr, [r for _, r in val_metrics], label="Val")
ax.set_title(f"Recall@{config_dict['K']}")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(log_epochs, log_ram)
ax.set_title("Process RAM Usage per Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("MB")
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(log_epochs, log_gpu, label="GPU Allocated")
ax.plot(log_epochs, log_peak, label="GPU Peak", linestyle="--")
ax.set_title("GPU VRAM per Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("MB")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
theory_labels = [k.replace("_MB", "").replace("_", " ") for k in theory.keys()]
theory_vals = list(theory.values())
bars = ax.barh(theory_labels, theory_vals)
ax.set_title("Theoretical Memory Breakdown")
ax.set_xlabel("MB")
for bar, val in zip(bars, theory_vals):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2, f"{val:.1f}", va="center", fontsize=8)
ax.grid(True, alpha=0.3, axis="x")

ax = axes[1, 2]
struct_labels = [
    "R_all CSR\n(CPU)",
    "R_train CSR\n(CPU)",
    "Split dict\ntrain",
    "Split dict\nval",
    "Split dict\ntest",
    "Embeddings\n(GPU params)",
    "Norm adj\n(GPU sparse)",
]
struct_vals = [
    r_all_mb,
    r_train_mb,
    train_mb,
    val_mb,
    test_mb,
    param_mb,
    adj_mb,
]
bars2 = ax.barh(struct_labels, struct_vals)
ax.set_title("Actual Data Structure Sizes")
ax.set_xlabel("MB")
for bar, val in zip(bars2, struct_vals):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2, f"{val:.1f}", va="center", fontsize=8)
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plot_path = base_path[config_dict["dataset_name"]] + "training_and_memory.png"
os.makedirs(base_path[config_dict["dataset_name"]], exist_ok=True)
# plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot path: {plot_path}")

# %%
# ---------------------------------------------------------------------------
# Final test evaluation
# ---------------------------------------------------------------------------

model.eval()
print(f"Training completed after {config_dict['epochs']} epochs")

snap_pre_test = snap("before test eval")

test_p, test_r = compute_metrics_sparse(
    model,
    eval_pos=test_pos,
    mask_pos=train_val_pos,
    n_users=n_users,
    K=config_dict["K"],
    eval_batch_size=config_dict["eval_batch_size"],
    device=device,
)

snap_post_test = snap("after test eval")

# Proper held-out BPR diagnostic:
# positives = test only
# negatives exclude all observed items (train + val + test)
samples_test = sample_bpr_triplets_sparse(
    pos_dict=test_pos,
    n_items=n_items,
    num_samples_per_user=max(1, config_dict["num_samples_per_user"] // 2),
    seed=config_dict["seed"] + 10,
    exclusion_dict=all_observed_pos,
).to(device)

with torch.no_grad():
    if len(samples_test) > 0:
        loss_test, reg_test = bpr_loss(
            model,
            samples_test[:, 0],
            samples_test[:, 1],
            samples_test[:, 2],
        )
        reg_test = reg_test * config_dict["weight_decay"]
        test_bpr = loss_test.item()
        test_reg = reg_test.item()
        test_total = test_bpr + test_reg
    else:
        test_bpr = test_reg = test_total = 0.0

print(
    f"Test BPR={test_bpr:.6f} | Reg={test_reg:.6f} | Total={test_total:.6f} | "
    f"P@{config_dict['K']}={test_p:.4f} R@{config_dict['K']}={test_r:.4f}"
)

# %%
# ---------------------------------------------------------------------------
# Final memory report
# ---------------------------------------------------------------------------

gpu_alloc_final, gpu_res_final, gpu_peak_final = get_gpu_mb()

print("\n" + "=" * 68)
print("FINAL MEMORY REPORT")
print("=" * 68)

all_snaps = [
    snap_start,
    snap_after_load,
    snap_after_split,
    snap_pre_model,
    snap_post_model,
    snap_post_optim,
    snap_pre_test,
    snap_post_test,
]
for s in all_snaps:
    print(s)

print("\n--- Peak GPU VRAM across full run ---")
print(f"Peak allocated: {gpu_peak_final:.2f} MB")
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(device).total_memory / 1024 ** 2
    print(f"GPU total VRAM: {total_vram:.2f} MB")
    print(f"VRAM utilisation: {100 * gpu_peak_final / total_vram:.1f}%")

print("\n--- Epoch memory stats ---")
if mem_log:
    ram_vals = [e["ram_mb"] for e in mem_log]
    gpu_vals = [e["gpu_alloc_mb"] for e in mem_log]
    peak_vals = [e["gpu_peak_mb"] for e in mem_log]
    print(f"RAM min={min(ram_vals):.1f} max={max(ram_vals):.1f} mean={np.mean(ram_vals):.1f} MB")
    print(f"GPU alloc min={min(gpu_vals):.1f} max={max(gpu_vals):.1f} mean={np.mean(gpu_vals):.1f} MB")
    print(f"GPU peak min={min(peak_vals):.1f} max={max(peak_vals):.1f} mean={np.mean(peak_vals):.1f} MB")

actual_total_mb = r_all_mb + r_train_mb + total_split_mb + param_mb + adj_mb
dense_equiv_gb = 3 * n_users * n_items * 4 / 1024 ** 3

print("\n--- Savings vs dense-mask version ---")
print(f"Dense masks (3 x U x I float32): {dense_equiv_gb:.1f} GB")
print(f"Actual tracked structures     : {actual_total_mb:.1f} MB")
if actual_total_mb > 0:
    print(f"Reduction factor              : ~{dense_equiv_gb * 1024 / actual_total_mb:.0f}x")
print("=" * 68 + "\n")

# %%
# ---------------------------------------------------------------------------
# Save results
# ---------------------------------------------------------------------------

os.makedirs(base_path[config_dict["dataset_name"]], exist_ok=True)

result = {
    "config": config_dict,
    "train_val_result": {
        "epochs_tracked": epochs_tracked,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
    },
    "test_result": {
        "BPR_loss": test_bpr,
        "Reg_loss": test_reg,
        "test_p": test_p,
        "test_r": test_r,
    },
    "timing": {
        "total_training_time_seconds": total_training_time,
        "propagation_time_seconds": total_prop_time,
        "score_loss_time_seconds": total_scoring_time,
        "forward_total_time_seconds": total_forward_time,
        "backward_step_time_seconds": total_backward_step_time,
        "evaluation_time_seconds": total_eval_time,
        "average_epoch_time_seconds": float(np.mean(epoch_times)),
    },
    "complexity": {
        "U": U,
        "I": I,
        "R_train": R_train_edges,
        "k": k_cfg,
        "L": L,
        "d": d,
        "training_complexity": "O(L|R|d + k|U|d)",
        "space_complexity": "O((|U|+|I|)d + |R|) [sparse]",
    },
    "memory": {
        "actual_MB": {
            "R_all_csr": r_all_mb,
            "R_train_csr": r_train_mb,
            "split_train": train_mb,
            "split_val": val_mb,
            "split_test": test_mb,
            "model_params": param_mb,
            "norm_adj_gpu": adj_mb,
        },
        "theoretical_MB": theory,
        "gpu_peak_MB": gpu_peak_final,
        "epoch_log": mem_log,
    },
    "model": model.state_dict(),
}

torch.save(result, output_file_name)
print(f"Saved to: {output_file_name}")

# %%
# ---------------------------------------------------------------------------
# Timing breakdown
# ---------------------------------------------------------------------------

print(f"Propagation time     = {total_prop_time:.2f}s")
print(f"Score/loss time      = {total_scoring_time:.2f}s")
print("-" * 45)
print(f"Forward total        = {total_forward_time:.2f}s")
print(f"Backward/step        = {total_backward_step_time:.2f}s")
print(f"Evaluation           = {total_eval_time:.2f}s")
print("-" * 45)
print(f"Fwd + Bwd + Eval     = {total_forward_time + total_backward_step_time + total_eval_time:.2f}s")
print(f"Total runtime        = {total_training_time:.2f}s")

PyTorch version: 2.8.0+cu126
CUDA available: True
Device: cuda
Output file: log/beauty2m/result_amazonBeauty2m_1layers_10sample_allusers.pkl
[MEM | startup (before anything)               ] RAM=   549.8 MB | GPU alloc=     0.0 MB | GPU peak=     0.0 MB
n_users:1,210,271  n_items:249,274  interactions:2,023,070
#Users: 1,210,271 | #Items: 249,274 | Interactions: 2,023,070
[MEM | after dataset load                      ] RAM=   711.4 MB | GPU alloc=     0.0 MB | GPU peak=     0.0 MB
R_all CSR actual RAM   : 20.05 MB
Dense equivalent would : 1123.9 GB
Train positives: 1,695,218
Val positives  : 152,937
Test positives : 174,915
Split dict RAM: train=6.47 MB | val=0.58 MB | test=0.67 MB
Total split RAM: 7.72 MB
[MEM | after train/val/test split              ] RAM=  1148.2 MB | GPU alloc=     0.0 MB | GPU peak=     0.0 MB

===== DATA / TRAINING STATS =====
Total users                 : 1,210,271
Total items                 : 249,274
Train users (active)        : 1,210,271
Train edges (R_trai

C:\Users\User\AppData\Local\Temp\ipykernel_29576\2971339136.py:402: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.where(deg > 0, deg ** (-0.5), 0.0).astype(np.float32)


[MEM | after model init                        ] RAM=  1748.2 MB | GPU alloc=   421.3 MB | GPU peak=   421.3 MB
[MEM | after optimizer init                    ] RAM=  1802.9 MB | GPU alloc=   421.3 MB | GPU peak=   421.3 MB

STATIC MEMORY BREAKDOWN
Embedding table (params)        :   356.33 MB
Norm adjacency (GPU sparse COO) :    64.67 MB
R_all CSR (CPU)                 :    20.05 MB
R_train CSR (CPU)               :    17.55 MB
Split dict total                :     7.72 MB

embeddings_MB                       :   356.33 MB
adj_values_MB                       :    12.93 MB
adj_indices_MB                      :    51.73 MB
propagation_MB                      :   712.67 MB
adam_moments_MB                     :   712.67 MB
gradients_MB                        :   356.33 MB
total_MB                            :  2202.67 MB

Training on epoch 0
Actual sampled triplets this epoch: 12,102,710
  Minibatch 1/2955 | BPR=0.655822 | Reg=0.096034
  Minibatch 101/2955 | BPR=0.208284 | Reg=0.143797
  

KeyboardInterrupt: 